# Loose HDF5 recipe

Companion notebook for the RDDAC [documentation](https://rddac.readthedocs.io). It is written to stand on its own: every step is annotated so the notebook reads top to bottom.

## Walkthrough

1. the `rddac download --extract --remove-zip` workflow that leaves loose `.h5` files on disk,
2. simulate it here by extracting three members of `sample.zip` into a throwaway directory,
3. inspect the loose layout,
4. read `process_parameters.csv` with pandas and filter the rows of interest,
5. open each loose `.h5` directly with `h5py`,
6. stream the loose layout with `rddac.streaming.iter_view` (this works — unlike the zip-only manifest path).

## Assumptions

- This notebook writes a throwaway copy into the system temp directory; the project-local `../data` stays untouched.
- `rddac.open_h5` and `RDDACDataset` walk the **zips** referenced by the manifest. Once `--remove-zip` deletes them, those two are gone for the extracted experiments — but `rddac.streaming.iter_view` recognises loose files natively (a deliberate difference from `ddacs`), so the view machinery keeps working.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import shutil
import tempfile
import zipfile
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

import rddac
print('rddac', rddac.__version__)

DATA_DIR  = Path('../data')
# DATA_DIR = Path('./data')   # uncomment instead when running from the repository root
assert (DATA_DIR / 'metadata.json').is_file(), f'{DATA_DIR} does not contain the dataset'
LOOSE_DIR = Path(tempfile.mkdtemp(prefix='rddac_loose_'))
print('loose layout goes to:', LOOSE_DIR)

rddac 0.1.0
loose layout goes to: /tmp/rddac_loose_w6y5flzl


## 1. Download with extract + remove

`rddac download --extract --remove-zip` fetches the bundle, unzips each archive next to the zip, and deletes the zip on success. The result is a directory of loose `NNNN.h5` files instead of `.zip` archives. Run this once from a shell:

```bash
rddac download --small --extract --remove-zip --out ./data -y
```

The zip members sit at the archive root, so extraction leaves the `.h5` files directly beside `metadata.json` and `process_parameters.csv`. (`rddac.streaming` also recognises an `h5/` subdirectory if you prefer to tidy them away.)

The dataset is already on disk in this repository, so instead of re-downloading, the next cell **simulates** the workflow: it extracts three members of `sample.zip` into the throwaway directory and copies the two index files alongside — byte-for-byte the layout the CLI command produces.

In [2]:
members = ['0000.h5', '0500.h5', '1002.h5']
with zipfile.ZipFile(DATA_DIR / 'h5' / 'sample.zip') as zf:
    for name in members:
        zf.extract(name, LOOSE_DIR)

shutil.copy(DATA_DIR / 'process_parameters.csv', LOOSE_DIR)
shutil.copy(DATA_DIR / 'metadata.json', LOOSE_DIR)
print('extracted', members)

extracted ['0000.h5', '0500.h5', '1002.h5']


## 2. Inspect the loose layout

After extraction, `metadata.json` and `process_parameters.csv` sit at the root next to the loose experiment files.

In [3]:
for entry in sorted(LOOSE_DIR.iterdir()):
    size_mb = entry.stat().st_size / 1e6
    print(f'  {entry.name:28s} {size_mb:8.1f} MB')

  0000.h5                          10.0 MB
  0500.h5                          10.0 MB
  1002.h5                          10.0 MB
  metadata.json                     0.0 MB
  process_parameters.csv            0.4 MB


## 3. Read `process_parameters.csv` with pandas

The CSV is the experiment index: one row per experiment id, with every process parameter exposed as a named column. Filter it in pandas before touching any HDF5 file.

In [4]:
params = pd.read_csv(LOOSE_DIR / 'process_parameters.csv')
print(f'rows: {len(params)}, columns: {list(params.columns)}')
print()
print(params.head(3).to_string())

rows: 9000, columns: ['index', 'experiment_id', 'category', 'geometry', 'blankholder_force', 'mean_punch_temp', 'oil_type', 'has_pointcloud', 'has_oil', 'split']

   index  experiment_id  category geometry  blankholder_force  mean_punch_temp oil_type  has_pointcloud  has_oil  split
0      0              1         0  concave                100             20.2   coarse            True     True    val
1      1              2         0  concave                100             20.3   coarse            True     True  train
2      2              3         0  concave                100             20.4   coarse            True     True    val


## 4. Iterate the loose files with `h5py`

Walk the (filtered) rows, build the zero-padded path, skip experiments whose `.h5` is missing locally, and open each one with `h5py.File`. Below: take the concave subset, then read the sheet-thickness traverse from every loose file that landed on disk. The raw traverse contains negative sensor-error spikes, so the summary masks `value <= 0` first (see the visualization notebook). With three extracted files the loop yields three lines.

In [5]:
concave = params.query("geometry == 'concave'")
print(f'concave experiments in CSV: {len(concave):>5d} of {len(params)}')

found = 0
for _, row in concave.iterrows():
    h5_path = LOOSE_DIR / f"{row['index']:04d}.h5"
    if not h5_path.is_file():
        continue
    with h5py.File(h5_path, 'r') as f:
        thickness = f['sheet_thickness/data'][:]
    valid = thickness[thickness[:, 1] > 0, 1]
    print(f"  experiment {row['index']:>4d}  oil_type={row['oil_type']:<7s} "
          f"sheet thickness {valid.min():7.1f} - {valid.max():7.1f} um  ({len(valid)} valid samples)")
    found += 1
print(f'\nopened {found} loose h5 file(s)')

concave experiments in CSV:  4500 of 9000
  experiment    0  oil_type=coarse  sheet thickness   985.9 -  1000.4 um  (206 valid samples)
  experiment  500  oil_type=fine    sheet thickness   950.1 -   988.7 um  (206 valid samples)
  experiment 1002  oil_type=medium  sheet thickness   983.2 -   998.4 um  (206 valid samples)



opened 3 loose h5 file(s)


## 5. Stream the loose layout with `iter_view`

`rddac.streaming.iter_view` builds a unified index at startup that recognises **both** loose `NNNN.h5` files (at the data-dir root or under `h5/`) and zipped archives — preferring loose files when both exist, because direct `h5py` reads skip the per-record `BytesIO` round trip. Pointing `data_dir` at the loose directory is therefore all it takes; views, slicing, `where=` filters, and the numpy export from the streaming notebook keep working unchanged.

The zip-backed single-file helper does *not* survive the transition: `rddac.open_h5` walks the manifest-mapped zips only, so on this directory it raises `FileNotFoundError` — reach for `h5py.File(path)` directly instead (step 4), the file name is just the zero-padded experiment id.

In [6]:
for rec in rddac.streaming.iter_view('force-curve', data_dir=LOOSE_DIR):
    print(f"  experiment {rec['_sim_id']:>4d}: force_data shape={rec['force_data'].shape}")

try:
    rddac.open_h5(0, data_dir=LOOSE_DIR)
except FileNotFoundError as e:
    print(f'\nopen_h5 on the loose layout: FileNotFoundError: {str(e)[:80]}...')

  experiment    0: force_data shape=(1140, 8)
  experiment  500: force_data shape=(1140, 8)
  experiment 1002: force_data shape=(1140, 8)

open_h5 on the loose layout: FileNotFoundError: 0000.h5 not found in any locally mapped zip under PosixPath('/tmp/rddac_loose_w6...


## 6. Clean up

The loose copy was a demonstration artifact; remove it so repeated runs start fresh.

In [7]:
shutil.rmtree(LOOSE_DIR)
print('removed', LOOSE_DIR)

removed /tmp/rddac_loose_w6y5flzl
